# Lecture 28 - Decision Trees, Random Forests, and Ensemble Methods

## Learning Objectives

- Train and visualise decision trees
- Interpret feature importance from trees
- Build random forests and tune key hyperparameters
- Understand bagging vs boosting conceptually
- Use GradientBoostingClassifier for improved performance

## Key Topics

- DecisionTreeClassifier / DecisionTreeRegressor
- Visualising trees and feature importance
- RandomForestClassifier / RandomForestRegressor
- Hyperparameters: n_estimators, max_depth, min_samples_split
- Bagging vs Boosting
- GradientBoostingClassifier

## Decision Trees

A **decision tree** splits the data recursively based on feature values, creating a flowchart-like structure. Each internal node tests a feature, each branch represents the outcome of the test, and each leaf holds a prediction.

Trees are highly interpretable — you can visualise them and literally read the decision rules. They handle both numeric and categorical data and capture non-linear relationships automatically. However, they tend to **overfit** unless pruned or constrained.

Key hyperparameters to control overfitting:
- `max_depth`: maximum depth of the tree
- `min_samples_split`: minimum samples required to split a node
- `min_samples_leaf`: minimum samples required in a leaf node

Scikit-learn provides `DecisionTreeClassifier` and `DecisionTreeRegressor` with identical APIs.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt

iris = load_iris()
X, y = iris.data, iris.target

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X, y)

plt.figure(figsize=(14, 8))
plot_tree(dt, filled=True, feature_names=iris.feature_names,
          class_names=iris.target_names, rounded=True)
plt.title("Decision Tree (max_depth=3) on Iris Dataset")
plt.show()

In [ ]:
# Feature importance from a decision tree
importance = pd.DataFrame({
    "feature": iris.feature_names,
    "importance": dt.feature_importances_
}).sort_values("importance", ascending=False)

print("Feature Importance:")
print(importance)

## Random Forest: Bagging + Random Feature Selection

A **Random Forest** builds many decision trees on bootstrapped samples of the data and averages their predictions. It also randomly selects a subset of features at each split, which decorrelates the trees.

The result is a model that:
- Retains most of the interpretability benefits (via feature importance)
- Dramatically reduces overfitting compared to a single tree
- Handles high-dimensional data well
- Requires minimal preprocessing (no scaling needed)

Key hyperparameters:
- `n_estimators`: number of trees (more is better, with diminishing returns)
- `max_depth`: constrain tree depth to control overfitting
- `min_samples_split`: prevent splits on very small subsets
- `max_features`: fraction of features to consider at each split

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_wine
import numpy as np

wine = load_wine()
X_w, y_w = wine.data, wine.target
X_tr, X_te, y_tr, y_te = train_test_split(X_w, y_w, test_size=0.3, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
print(f"Random Forest accuracy: {rf.score(X_te, y_te):.3f}")

In [ ]:
# Hyperparameter tuning: n_estimators and max_depth
for n in [10, 50, 100, 200]:
    for d in [None, 5, 10]:
        rf_tune = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42)
        rf_tune.fit(X_tr, y_tr)
        acc = rf_tune.score(X_te, y_te)
        print(f"n_estimators={n:3d}, max_depth={str(d):4s}  accuracy={acc:.3f}")

In [ ]:
# Feature importance analysis
import pandas as pd
import matplotlib.pyplot as plt

feat_imp = pd.DataFrame({
    "feature": wine.feature_names,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 5))
plt.barh(feat_imp["feature"], feat_imp["importance"])
plt.xlabel("Feature Importance")
plt.title("Random Forest Feature Importance - Wine Dataset")
plt.gca().invert_yaxis()
plt.show()

## Bagging vs Boosting

**Bagging** (Bootstrap Aggregating) trains many models in parallel on different bootstrap samples and averages their predictions. Random Forest is the most famous example. Bagging reduces variance without increasing bias significantly.

**Boosting** trains models sequentially, where each new model focuses on the mistakes made by the previous one. Models are added to correct the errors of the ensemble. Boosting reduces both bias and variance, but can overfit if not carefully regularised.

Common boosting algorithms include AdaBoost, Gradient Boosting, and XGBoost. In practice, gradient boosting often produces state-of-the-art results on tabular data, but requires more hyperparameter tuning than random forests.

In [ ]:
# Basic comparison: RandomForest (bagging) vs GradientBoosting (boosting)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)

for name, model in [("RandomForest", rf_model), ("GradientBoosting", gb_model)]:
    scores = cross_val_score(model, X_w, y_w, cv=5)
    print(f"{name:20s}  CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")

In [ ]:
# GradientBoosting with different learning rates
for lr in [0.01, 0.1, 0.5, 1.0]:
    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, random_state=42)
    scores = cross_val_score(gb, X_w, y_w, cv=5)
    print(f"learning_rate={lr:4.2f}  CV accuracy: {scores.mean():.3f}")

## Data Science Connection

Ensemble methods — particularly random forests and gradient boosting — dominate tabular data competitions on Kaggle and are widely used in industry. Feature importance from tree-based models is one of the most powerful tools for understanding your data: it tells you which features actually drive predictions. When you need a strong out-of-the-box model with minimal preprocessing, reach for a random forest. When you need to squeeze out every last percentage point of accuracy, gradient boosting is your friend.